<a href="https://colab.research.google.com/github/BenFreilichUVA/ds2002-fa26/blob/main/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last_problem = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            last_problem = repr(e)
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                raise ValueError(f'{r.status_code} (our bug, not retrying): {r.text[:200]}')
            last_problem = f'HTTP {r.status_code}'
        if attempt < tries - 1:
            time.sleep(2 ** attempt)
    raise RuntimeError(f'get_weather failed after {tries} tries: {last_problem}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
Dates = ['2026-09-05', '2026-10-03']
used_fallback = False
try:
  FullRange = get_weather(min(Dates), max(Dates))
  wx = FullRange[FullRange['time'].isin(Dates)].copy()
  print(f'Pulled {len(FullRange)} days from the API, kept {len(wx)} of the days.')
  pulled_at = pd.Timestamp.now(tz='America/New_York')
except RuntimeError as e:
  used_fallback = True
  print(f'The API was unavailable to reach, we must fall back to the sample data: {e}')



Pulled 29 days from the API, kept 2 of the days.


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'

wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].reset_index(drop=True)

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
WetterDay = wx.loc[wx['precip_mm'].idxmax()]
DrierDay  = wx.loc[wx['precip_mm'].idxmin()]
Difference   = WetterDay['precip_mm'] - DrierDay['precip_mm']

print(f"{WetterDay['date']:%B %-d} was the wetter day, with {WetterDay['precip_mm']:.1f} mm "
      f"of rain, versus {DrierDay['precip_mm']:.1f} mm on "
      f"{DrierDay['date']:%B %-d}, which is a difference of {Difference:.1f} mm of rain (source: {WetterDay['source']}).")

October 3 was the wetter day, with 14.7 mm of rain, versus 2.5 mm on September 5, which is a difference of 12.2 mm of rain (source: open-meteo).


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:

LAT = 38.03
LON = -78.51
print("Citation for data:")
print(f'Endpoint: {BASE}')
print(f'Coordinates: {LAT}, {LON}')
print(f'Requested: {min(Dates)} to {max(Dates)}')

if used_fallback:
    print('Data: fallback sample')
else:
    print('Data: Real data from open-meteo')
    print(f'Pulled: {pulled_at:%Y-%m-%d %H:%M:%S %Z}')


Citation for data:
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Requested: 2026-09-05 to 2026-10-03
Data: Real data from open-meteo
Pulled: 2026-10-09 15:09:04 EDT


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If an empty DataFrame was returned, we may see that rain is entirely NaN for the entirety of the DataFrame, as the empty data is still merged onto the data for sales. The chart would thus show no relationship between rain and revenue, and to an unobservant reader, it is possible to assume that there really is just no relationship and move on.

Additionally, it is possible there will be a function to fill all NaN values with either a mean value, a set number, or just zero. If the person pulling the DataFrame isn't aware that this exists, they may assume that is the actual weather pattern, no matter how irregular it seems.

It is unlikely someone would catch this because the program will never explicitly tell them there is a problem. Raising the error to the user informs them that there is an explicit issue that needs to be addressed for the integrity of the project, while simply returning the empty DataFrame will allow the process to go on as normal.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch

Cities = {
    'Charlottesville': (38.03, -78.51),
    'Orlando': (28.54, -81.38),
}
def get_city_weather(city, lat, lon, dates):
  FullRange = get_weather(min(dates), max(dates), lat=lat, lon=lon)
  CondensedFrame = (
      FullRange[FullRange['time'].isin(dates)]
      .rename(columns={
          'time': 'date',
          'temperature_2m_max': 'temp_max',
          'precipitation_sum': 'precip_mm',
      }).assign(date = lambda d: pd.to_datetime(d['date']), city = city, source = 'open-meteo',
      )
      [['city', 'date', 'temp_max', 'precip_mm', 'source']]
  )
  return CondensedFrame
Individual_Rows = [get_city_weather(city, lat, lon, Dates) for city, (lat, lon) in Cities.items()]
wx_two_cities = pd.concat(Individual_Rows, ignore_index=True)
wx_two_cities


,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
